# LightGBM：leaf-wise 原生类别 Boosting 挑战者

本 Notebook 使用与共同输入 Logistic、随机森林和 CatBoost 完全相同的 126 个公共特征和固定 Train/Selection 切分。数值缺失由 LightGBM 原生处理；15 个物理类别变量仅由 Train 定义类别集合，Selection 未见类别显式标记为 `__UNSEEN__`。候选方案覆盖叶节点容量、深度、最小叶样本、学习率、L1/L2 正则、行列抽样、最小分裂收益与类别平滑。Selection 仅用于 early stopping、锁模和与共同输入 Logistic 的配对比较，Test 保持封存。锁模后的 Tree SHAP 只在 Selection 分层抽样上计算。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.comparison import load_locked_selection_prediction
from credit_risk.development import load_development_data
from credit_risk.models.lightgbm import run_lightgbm
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
development = load_development_data(PROJECT_ROOT)
assert not hasattr(development, 'X_test')
assert not hasattr(development, 'y_test')
reference_selection_prediction = load_locked_selection_prediction(
    development, 'common_logistic'
)
display(development.split_overview)
print('公共特征数:', len(development.feature_columns))
print('已对齐的共同输入 Logistic Selection 预测数:', len(reference_selection_prediction))

,sample,n,bad_rate
0,Train,215257,0.080727
1,Selection,46127,0.080734
2,Test (sealed),46127,NaN


公共特征数: 126
已对齐的共同输入 Logistic Selection 预测数: 46127


In [3]:
search_settings = development.configs['model_search_spaces']['lightgbm']
display(pd.DataFrame(search_settings['candidate_profiles']))
print('基础迭代上限:', search_settings['max_iterations'])
print('边界延长上限:', search_settings['extension_max_iterations'])
print('Early stopping patience:', search_settings['early_stopping_rounds'])

,name,num_leaves,max_depth,min_child_samples,learning_rate,reg_lambda,reg_alpha,subsample,colsample_bytree,min_split_gain,cat_smooth
0,conservative_smooth,15,5,300,0.02,10.0,0.0,0.7,0.8,0.00,20.0
1,shallow_stable,31,6,300,0.02,5.0,0.0,0.8,0.9,0.00,20.0
2,legacy_balanced,31,6,200,0.03,5.0,0.0,0.8,0.9,0.00,10.0
3,regularized_balanced,31,6,300,0.03,10.0,0.5,0.8,0.9,0.05,20.0
4,leaf_rich_regularized,63,7,300,0.03,10.0,1.0,0.8,0.8,0.05,20.0
5,faster_strong_l2,31,6,300,0.05,10.0,1.0,0.8,0.9,0.05,20.0
6,wide_conservative,63,7,500,0.02,20.0,1.0,0.7,0.8,0.10,50.0


基础迭代上限: 3000
边界延长上限: 5000
Early stopping patience: 150


In [4]:
result = run_lightgbm(
    development,
    reference_selection_prediction=reference_selection_prediction,
    register_lock=True,
)
print('LightGBM 搜索、固定迭代重训与锁模完成。')

LightGBM candidate 1 [base_search/conservative_smooth] Selection AR=0.531868, gap=0.080665, trees=1237/1387, fit=16.0s


LightGBM candidate 2 [base_search/shallow_stable] Selection AR=0.533357, gap=0.150746, trees=1183/1333, fit=22.2s


LightGBM candidate 3 [base_search/legacy_balanced] Selection AR=0.532244, gap=0.122101, trees=574/724, fit=12.8s


LightGBM candidate 4 [base_search/regularized_balanced] Selection AR=0.530832, gap=0.107964, trees=537/687, fit=12.3s


LightGBM candidate 5 [base_search/leaf_rich_regularized] Selection AR=0.530441, gap=0.154427, trees=475/625, fit=13.7s


LightGBM candidate 6 [base_search/faster_strong_l2] Selection AR=0.531122, gap=0.116969, trees=364/514, fit=9.5s


LightGBM candidate 7 [base_search/wide_conservative] Selection AR=0.531323, gap=0.150615, trees=911/1061, fit=21.7s


LightGBM 搜索、固定迭代重训与锁模完成。


In [5]:
display(result.search_results)
display(result.metrics)
display(result.uplift_vs_logistic)
display(result.feature_importance.head(30))
display(result.shap_importance.head(30))
display(result.selection_deciles)
display(result.locked_config)
print('锁模产物:', result.run_directory)

,candidate,stage,profile,max_iterations,attempted_iterations,best_iteration,reached_iteration_boundary,num_leaves,max_depth,min_child_samples,...,selection_ar,ar_gap,train_ks,selection_ks,selection_ap,selection_logloss,capacity,fit_seconds,parameters_json,eligible_for_lock
0,2,base_search,shallow_stable,3000,1333,1183,False,31,6,300,...,0.533357,0.150746,0.523150,0.389967,0.257149,0.243418,88.01520,22.223525,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 20.0, ...",True
1,3,base_search,legacy_balanced,3000,724,574,False,31,6,200,...,0.532244,0.122101,0.495997,0.391313,0.254591,0.243741,64.05840,12.788074,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 10.0, ...",True
2,1,base_search,conservative_smooth,3000,1387,1237,False,15,5,300,...,0.531868,0.080665,0.461769,0.390241,0.257461,0.243579,34.63600,15.962686,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 20.0, ...",True
3,7,base_search,wide_conservative,3000,1061,911,False,63,7,500,...,0.531323,0.150615,0.520529,0.394894,0.255495,0.243761,64.28016,21.651785,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 50.0, ...",True
4,6,base_search,faster_strong_l2,3000,514,364,False,31,6,300,...,0.531122,0.116969,0.491052,0.389735,0.255131,0.243821,27.08160,9.536955,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 20.0, ...",True
5,4,base_search,regularized_balanced,3000,687,537,False,31,6,300,...,0.530832,0.107964,0.483580,0.389868,0.256502,0.243730,39.95280,12.334686,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 20.0, ...",True
6,5,base_search,leaf_rich_regularized,3000,625,475,False,63,7,300,...,0.530441,0.154427,0.521605,0.392555,0.256042,0.243806,63.84000,13.672703,"{""boosting_type"": ""gbdt"", ""cat_smooth"": 20.0, ...",True


,sample,n,bad_rate,AUC,AR,KS,AP,Logloss
0,Train,215257,0.080727,0.806267,0.612534,0.461769,0.312560,0.231308
1,Selection,46127,0.080734,0.765934,0.531868,0.390241,0.257461,0.243579


,model,reference_model,metric,difference,lower,upper,bootstrap_se,confidence_level,bootstrap_replicates,higher_is_better
0,lightgbm,common_logistic,AUC,0.017791,0.014467,0.021549,0.001757,0.95,500,True
1,lightgbm,common_logistic,AR,0.035583,0.028933,0.043099,0.003515,0.95,500,True
2,lightgbm,common_logistic,KS,0.023763,0.015312,0.035061,0.005046,0.95,500,True
3,lightgbm,common_logistic,AP,0.022031,0.015868,0.028534,0.003195,0.95,500,True
4,lightgbm,common_logistic,Logloss,-0.005096,-0.006066,-0.004189,0.000475,0.95,500,False


,feature,total_gain,split_count,gain_pct
0,EXT_SOURCE_3,90889.887794,920,0.208237
1,EXT_SOURCE_2,83236.178406,736,0.190702
2,ORGANIZATION_TYPE,40888.565337,2708,0.093679
3,EXT_SOURCE_1,31365.510540,885,0.071861
4,CREDIT_ANNUITY_RATIO,29544.022735,1236,0.067688
5,GOODS_CREDIT_RATIO,15116.603284,454,0.034634
6,OCCUPATION_TYPE,14647.559052,966,0.033559
7,EMPLOYED_YEARS,13659.071999,362,0.031294
8,AGE_YEARS,9852.225103,646,0.022572
9,AMT_ANNUITY,7513.562031,402,0.017214


,feature,mean_abs_shap
0,EXT_SOURCE_3,0.344847
1,EXT_SOURCE_2,0.296161
2,EXT_SOURCE_1,0.181273
3,CREDIT_ANNUITY_RATIO,0.132738
4,GOODS_CREDIT_RATIO,0.106151
5,EMPLOYED_YEARS,0.089403
6,ORGANIZATION_TYPE,0.088496
7,AMT_ANNUITY,0.087418
8,OWN_CAR_AGE,0.079146
9,AMT_GOODS_PRICE,0.077478


,risk_group,rank_min,rank_max,n,bad_n,bad_rate,probability_min,probability_max,lift,cumulative_bad_capture
0,1,1,4613,4613,1288,0.279211,0.182265,0.770628,3.458422,0.345865
1,2,4614,9226,4613,680,0.147409,0.118335,0.182259,1.825875,0.528464
2,3,9227,13839,4613,460,0.099718,0.086049,0.118333,1.235151,0.651987
3,4,13840,18451,4612,386,0.083695,0.065693,0.086048,1.036677,0.755639
4,5,18452,23064,4613,291,0.063083,0.051494,0.065692,0.781367,0.833781
5,6,23065,27677,4613,218,0.047258,0.040720,0.051494,0.585354,0.892320
6,7,27678,32289,4612,148,0.032090,0.032199,0.040714,0.397482,0.932062
7,8,32290,36902,4613,122,0.026447,0.024835,0.032198,0.327583,0.964823
8,9,36903,41515,4613,82,0.017776,0.017623,0.024835,0.220179,0.986842
9,10,41516,46127,4612,49,0.010624,0.003533,0.017621,0.131599,1.000000


{'model_name': 'lightgbm',
 'experiment_track': 'controlled',
 'data_sha256': '52e96b895b1112e1c853f670e58372719c8441c5ed1c57ac2f7fad559d784f5f',
 'split_sha256': '4ae4ceee661ba8c210fa0a1cf950448799a0aef468659e6d2590c1bb4e8a3835',
 'feature_policy_version': 1,
 'training_sample': 'Train only',
 'selection_role': 'early stopping, candidate selection, and paired comparison only',
 'test_accessed': False,
 'parameters': {'objective': 'binary',
  'metric': 'auc',
  'boosting_type': 'gbdt',
  'n_estimators': 1237,
  'num_leaves': 15,
  'max_depth': 5,
  'min_child_samples': 300,
  'learning_rate': 0.02,
  'reg_lambda': 10.0,
  'reg_alpha': 0.0,
  'subsample': 0.7,
  'colsample_bytree': 0.8,
  'min_split_gain': 0.0,
  'cat_smooth': 20.0,
  'subsample_freq': 1,
  'max_bin': 255,
  'feature_pre_filter': False,
  'deterministic': True,
  'force_col_wise': True,
  'use_missing': True,
  'zero_as_missing': False,
  'importance_type': 'gain',
  'random_state': 42,
  'n_jobs': -1,
  'verbosity': -1

锁模产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/lightgbm/20260831T065642_605349Z


In [6]:
registry_path = PROJECT_ROOT / development.configs['experiment']['test_policy']['registry_path']
registry = json.loads(registry_path.read_text(encoding='utf-8'))
assert registry['models']['lightgbm']['locked'] is True
assert registry['models']['lightgbm']['config']['test_accessed'] is False
assert registry['suite_locked'] is False
print('LightGBM 已锁定；XGBoost、MLP 尚未全部登记，Test 继续封存。')

LightGBM 已锁定；XGBoost、MLP 尚未全部登记，Test 继续封存。
